<img src="https://github.com/pmservice/ai-openscale-tutorials/raw/master/notebooks/images/banner.png" align="left" alt="banner">

# Drift v2 Baseline Archive Computation for GenAI Batch Subscriptions
## IBM Watson OpenScale - IBM Cloud Pak for Data v5.4

Generates the **Drift v2 baseline archive** for a GenAI Batch (prompt asset) subscription
by submitting `DriftV2ConfigurationJob` to the CPD IAE Spark cluster.
Supports both storage backends:

| Route | Curated table source |
| :- | :- |
| **JDBC** | DB2 / PostgreSQL via `JdbcStorageUtils` |
| **Object Storage** | IBM COS / S3 Iceberg via `ObjectStorageUtils` |

**Steps:**
1. [Install Dependencies](#1-install)
2. [Subscription Details](#2-subscription)
3. [Storage Connection](#3-storage) - fill in JDBC **or** Object Storage block
4. [Generate Baseline Archive](#4-generate)
5. [Verify & Download](#5-verify)

<a id="1-install"></a>
## 1. Install Dependencies

> **Restart the kernel** after this cell finishes, then continue from cell 2.

In [ ]:
# On zLinux clusters install scikit-learn via conda BEFORE ibm-metrics-plugin:
# !conda install scikit-learn=1.6.1

!pip install --no-warn-conflicts "ibm-metrics-plugin[notebook]~=5.4.0" nest_asyncio pyarrow | tail -n 1

# On RT 24.1 environments uncomment and run:
# !pip install --upgrade "matplotlib~=3.10.1" transformers numexpr bottleneck opencv-python --no-cache | tail -n 1

In [ ]:
VERSION = "2.0"
print("Notebook version:", VERSION)

Notebook version: 1.0


<a id="2-subscription"></a>
## 2. Subscription Details

| Parameter | Where to find it |
| :- | :- |
| `pta_id` | `asset.asset_id` on the subscription resource — the landing table is `landing_<pta_id>` |
| `problem_type` | `asset.problem_type` on the subscription resource |
| `archive_output_dir` | Writable local path for the `.tar.gz` output |
| `baseline_max_samples` | Maximum rows stored in the curated table before archive generation begins (default = 10000) |
| `min_samples` | Minimum rows the drift evaluator needs (default: 10) |
| `poll_interval` | Seconds to wait between curated-table row-count re-reads while waiting for `min_samples` rows to arrive |

In [ ]:
common_params = {
    "pta_id":               "<PTA_ID>",  
    "problem_type":         "<problem_type",      
    "archive_output_dir":   "<output_path>",  
    "baseline_max_samples": 10000,  
    "min_samples":          10,     
    "poll_interval":        100,   
}

<a id="3-storage"></a>
## 3. Storage Connection

Fill in **one** block below to match your landing table storage type, then set `storage_connection` at the bottom.
Connection values are on `data_sources[type=landing]` of the subscription resource.

In [ ]:
# A - JDBC (DB2 / PostgreSQL) 

jdbc_storage_connection = {
    "connection": {
        "location_type": "jdbc",
        "db_driver":     "",
        "use_ssl":       False,
        "jdbc_url": "<jdbc_url>",
    },
    "credentials": {
        "username": "<username>",
        "password": "<pwd>",
    },
    "database_name": "<database_name>",   # mandatory, not derived from URL
    "schema_name":   "<schema_name",
    "type": "jdbc",
}

In [ ]:
# B — Object Storage (IBM COS / S3)

object_storage_connection = {
    "connection": {
        "location_type": "object_storage",
        "bucket_name":   "<to_be_edited>",
    },
    "credentials": {
        "access_key_id":     "<to_be_edited>",
        "secret_access_key": "<to_be_edited>",
        "endpoint":          "<to_be_edited>",  # e.g. https://s3.us-south.cloud-object-storage.appdomain.cloud
    },
    "type": "object_storage",
    "description": "",
    "name": "<OBJECT_STORAGE_CONNECTION_NAME>",
    "status": {"state": "active"},
}

In [ ]:
# set to jdbc_storage_connection or object_storage_connection
storage_connection = jdbc_storage_connection   # <to_be_edited>

# spark connection definition
spark_connection = {
    "connection": {
        "display_name": "<IAE_DISPLAY_NAME>",
        "endpoint": "<SPARK_ENDPOINT>",
        "location_type": "cpd_iae",
        "volume": "<VOLUME>",
    },
    "credentials": {
        "apikey": "<CPD_APIKEY>",
        "username": "<CPD_USERNAME>",
    },
    "description": "",
    "name": "Spark Engine - GenAI Batch Setup",
    "status": {"state": "active"},
    "type": "spark",
}

<a id="4-generate"></a>
## 4. Generate Baseline Archive

Submits `DriftV2ConfigurationJob` to the IAE Spark cluster, polls until completion,
then downloads the archive from the IAE volume:

1. Reads `curated_<pta_id>` from the configured storage backend (JDBC or Object Storage).
2. Runs `DriftMetricsEvaluator.fit()` on the curated data → `.tar.gz` archive on the IAE volume.
3. Downloads the archive locally and creates a download link.

In [ ]:
%%time
from ibm_metrics_plugin.common.utils.genai_batch_configuration_utility import ConfigurationUtilityGenAIBatch

config_utility = ConfigurationUtilityGenAIBatch(
    common_params=common_params,
    storage_connection=storage_connection,
    spark_connection_info=spark_connection,
)
config_utility.generate_batch_configuration()

<a id="5-verify"></a>
## 5. Verify & Download

Deserialises the archive and confirms it is valid before you upload it to the monitor.

In [ ]:
from ibm_metrics_plugin.metrics.drift_v2.utils.drift_utils import load_archive

loaded = load_archive(config_utility._archive_path)
ds     = config_utility._baseline_dataset  # None on the IAE path

_ac = getattr(loaded, "advanced_controls", {}) or {}
assert _ac.get("enable_drift_v2_prompt_batch") is True, \
    f"enable_drift_v2_prompt_batch not True in archive: {_ac}"

# Cross-check loaded archive against in-memory dataset (local path only)
if ds is not None:
    assert loaded.id == ds.id,               f"id mismatch: {loaded.id!r} != {ds.id!r}"
    assert str(loaded.type) == str(ds.type), f"type mismatch: {loaded.type} != {ds.type}"

print("PASS")
print("id                :", loaded.id)
print("type              :", loaded.type)
print("columns           :", sorted((getattr(loaded, 'columns', {}) or {}).keys()))
print("advanced_controls :", _ac)
print("archive path      :", config_utility._archive_path)

In [121]:
# Re-download the archive without regenerating it
config_utility.create_download_link()